# Track C — 03. Mutations Scoring

> **TEMP SCORING — p0 and k values are placeholders pending calibration against CRISPR/GDSC anchors. Driver boost b=0.15 is locked. Logged in DECISIONS.md as pending.**

Converts `cleaned_track_data/mutations_collapsed.parquet` into a per-`(ensg_id, model_id)`
probability score using a noisy-OR Hill-function model.

**Pipeline:**
1. Hill-function components from `max_vep_rank`, `max_pathogenicity`, `variant_burden`
2. Noisy-OR base score (full: quality + burden; split: quality-only and burden-only)
3. Additive driver boost from `oncogene_hit | tsg_hit` (b = 0.15, locked)
4. Max-aggregate to one row per `(ensg_id, model_id)` if duplicates exist

**Output:** `cleaned_track_data/mutations_scores.parquet`  
**Columns:**
- `ensg_id`, `model_id`
- `p_mutation` — full score: Noisy-OR(vep, path, burden) + driver boost  *(display / ranking)*
- `p_mutation_quality` — Noisy-OR(vep, path) only, no burden, no boost  *(for signature discount)*
- `p_mutation_burden` — Hill(variant_burden) alone  *(for signature discount)*

The combine step (05) applies the mutation signature discount as:
`p_mutation_adj = Noisy-OR(p_mutation_quality, m_mut * p_mutation_burden) + driver_boost`

In [ ]:
import os
import sys

_notebook_dir = os.path.dirname(os.path.abspath('__file__'))
# src/Track - C/ -> up one level to src/, then into scripts/
sys.path.insert(0, os.path.join(_notebook_dir, '..', 'scripts'))

import numpy as np
import pandas as pd
from data_utils import REF

PROJECT_ROOT = os.path.abspath(os.path.join(REF, '..'))
DATA_DIR     = os.path.join(PROJECT_ROOT, 'cleaned_track_data')

IN_MUT    = os.path.join(DATA_DIR, 'mutations_collapsed.parquet')
IN_GENE   = os.path.join(REF, 'gene_lookup.parquet')
IN_CELL   = os.path.join(REF, 'cell_line_lookup.parquet')
OUT_PATH  = os.path.join(DATA_DIR, 'mutations_scores.parquet')

# --- Hill-function hyperparameters (TEMP — pending CRISPR/GDSC calibration) ---
P0_VEP    = 3.0;   K_VEP    = 2.0
P0_PATH   = 0.5;   K_PATH   = 2.0
P0_BURDEN = 3.0;   K_BURDEN = 1.5
DRIVER_BOOST = 0.15  # locked

print('IN_MUT:  ', IN_MUT)
print('OUT_PATH:', OUT_PATH)

In [ ]:
mut  = pd.read_parquet(IN_MUT)
glk  = pd.read_parquet(IN_GENE)

print(f'mutations_collapsed: {len(mut):,} rows x {mut.shape[1]} cols')
print(f'gene_lookup:         {len(glk):,} rows')
print(f'distinct (ensg_id, model_id) pairs: {mut[["ensg_id", "model_id"]].drop_duplicates().shape[0]:,}')

## 1. Hill function + per-row probability components

```
Hill(x, p0, k) = x^k / (x^k + p0^k)

p_vep    = Hill(max_vep_rank,      p0=3,   k=2)
p_path   = Hill(max_pathogenicity, p0=0.5, k=2)
p_burden = Hill(variant_burden,    p0=3,   k=1.5)
```

NaN inputs propagate to NaN components; `fillna(0)` before noisy-OR so missing
evidence contributes nothing rather than producing NaN scores.

In [ ]:
def hill(x, p0, k):
    """Sigmoid-shaped Hill function mapping x -> (0, 1)."""
    xk = np.power(x.clip(lower=0), k)
    return xk / (xk + p0 ** k)


df = mut[['ensg_id', 'model_id', 'max_vep_rank', 'max_pathogenicity',
          'variant_burden', 'multi_hit_high_impact',
          'oncogene_hit', 'tsg_hit']].copy()

df['p_vep']    = hill(df['max_vep_rank'].astype(float),    P0_VEP,    K_VEP)
df['p_path']   = hill(df['max_pathogenicity'].astype(float), P0_PATH,  K_PATH)
df['p_burden'] = hill(df['variant_burden'].astype(float),  P0_BURDEN, K_BURDEN)

print('p_vep   describe:')
print(df['p_vep'].describe().round(4))
print()
print('p_path  describe (NaN = no pathogenicity score):')
print(df['p_path'].describe().round(4))
print()
print('p_burden describe:')
print(df['p_burden'].describe().round(4))

## 2. Noisy-OR base score + driver boost

```
p_base     = 1 - (1 - p_vep) * (1 - p_path) * (1 - p_burden)
driver_flag = (oncogene_hit | tsg_hit).astype(int)
p_mutation  = min(p_base + 0.15 * driver_flag, 1.0)
```

Then max-aggregate over any duplicate `(ensg_id, model_id)` pairs.

In [ ]:
# Fill NaN components with 0 — missing evidence contributes nothing to the OR
p_vep    = df['p_vep'].fillna(0.0)
p_path   = df['p_path'].fillna(0.0)
p_burden = df['p_burden'].fillna(0.0)

p_base = 1.0 - (1.0 - p_vep) * (1.0 - p_path) * (1.0 - p_burden)

driver_flag = (df['oncogene_hit'].fillna(False) | df['tsg_hit'].fillna(False)).astype(int)

df['p_mutation']         = (p_base + DRIVER_BOOST * driver_flag).clip(upper=1.0)
# Split components needed by the combine step to apply signature discount correctly:
#   m_mut scales burden only, not quality — so they must be kept separate.
df['p_mutation_quality'] = 1.0 - (1.0 - p_vep) * (1.0 - p_path)  # quality only, no burden
df['p_mutation_burden']  = p_burden                                 # burden alone

print('p_mutation describe:')
print(df['p_mutation'].describe().round(4))
print()
print(f'rows with driver boost: {(driver_flag == 1).sum():,}')
print(f'rows capped at 1.0:     {(df["p_mutation"] == 1.0).sum():,}')
print()
print('p_mutation_quality describe:')
print(df['p_mutation_quality'].describe().round(4))
print()
print('p_mutation_burden describe:')
print(df['p_mutation_burden'].describe().round(4))

In [ ]:
# Max-aggregate to one row per (ensg_id, model_id)
scores = (
    df.groupby(['ensg_id', 'model_id'], sort=False)[
        ['p_mutation', 'p_mutation_quality', 'p_mutation_burden']
    ]
    .max()
    .reset_index()
)

print(f'input rows:  {len(df):,}')
print(f'output rows: {len(scores):,}  (one per ensg_id / model_id pair)')
print(f'distinct genes:      {scores["ensg_id"].nunique():,}')
print(f'distinct cell lines: {scores["model_id"].nunique():,}')

## 3. Write output

In [ ]:
scores[['ensg_id', 'model_id', 'p_mutation', 'p_mutation_quality', 'p_mutation_burden']].to_parquet(OUT_PATH, index=False)
print(f'written: {OUT_PATH}')
print(f'rows:    {len(scores):,}')
print(f'columns: {list(scores.columns)}')

## 4. Validation spot-check

Resolve canonical model IDs from `cell_line_lookup` then look up known driver anchors.
All three should return high `p_mutation` (> 0.7) where the pair exists in the data.

| Gene | Cell line | Expected |
|------|-----------|----------|
| BRAF (ENSG00000157764) | A375 | > 0.7 |
| EGFR (ENSG00000146648) | A431 | > 0.7 |
| KRAS (ENSG00000133703) | HCT116 | > 0.7 |

In [ ]:
clk = pd.read_parquet(IN_CELL)

# Resolve cell-line names to model_id (cell_line_name is lowercase in lookup)
name_to_model = (
    clk[['model_id', 'cell_line_name']]
      .dropna(subset=['cell_line_name'])
      .drop_duplicates(subset=['cell_line_name'])
      .set_index('cell_line_name')['model_id']
)

ANCHORS = [
    ('ENSG00000157764', 'BRAF',  'a-375',   'A375'),
    ('ENSG00000146648', 'EGFR',  'a-431',   'A431'),
    ('ENSG00000133703', 'KRAS',  'hct 116', 'HCT116'),
]

scores_idx = scores.set_index(['ensg_id', 'model_id'])['p_mutation']

print(f'{"Gene":<6}  {"Cell line":<8}  {"model_id":<12}  {"p_mutation":>10}  {"status"}')
print('-' * 65)
for ensg, gene, lookup_name, display_name in ANCHORS:
    model_id = name_to_model.get(lookup_name, None)
    if model_id is None:
        print(f'{gene:<6}  {display_name:<8}  {"NOT IN LOOKUP":<12}  {"N/A":>10}  SKIP')
        continue
    score = scores_idx.get((ensg, model_id), float('nan'))
    if np.isnan(score):
        status = 'MISSING FROM DATA'
    elif score > 0.7:
        status = 'OK (> 0.7)'
    else:
        status = f'LOW — check inputs'
    print(f'{gene:<6}  {display_name:<8}  {model_id:<12}  {score:>10.4f}  {status}')